# Week 3 — Data Preprocessing

This notebook prepares the CRMLS SingleFamilyResidence sold data for AVM modeling.

Key principles:
- chronological train/validation/test split;
- no random split;
- no target leakage;
- train-only fitting for imputation, encoding, scaling, and outlier thresholds;
- the final cleaned CSV remains human-readable, while model-time transformations are stored in a scikit-learn pipeline.


In [ ]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


## 1. Install and import packages

In [ ]:
!pip -q install geopandas pyogrio

import os
import glob
import json
import joblib
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import geopandas as gpd

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 150)

RANDOM_SEED = 42
TARGET = "ClosePrice"
DATE_COL = "CloseDate"

DATA_DIR = Path("/content/drive/MyDrive/IDX_Exchange_Su26/data")
RAW_DIR = DATA_DIR / "california"
OUTPUT_DIR = DATA_DIR / "week3_outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

MONTHS = [
    "202501", "202502", "202503", "202504", "202505", "202506",
    "202507", "202508", "202509", "202510", "202511", "202512",
    "202601", "202602", "202603", "202604", "202605"
]

TRAINING_WINDOW_CANDIDATES = [3, 6, 9, 12, 15]
DEFAULT_TRAINING_WINDOW = 12

print("RAW_DIR:", RAW_DIR)
print("OUTPUT_DIR:", OUTPUT_DIR)

RAW_DIR: /content/drive/MyDrive/IDX_Exchange_Su26/data/california
OUTPUT_DIR: /content/drive/MyDrive/IDX_Exchange_Su26/data/week3_outputs


## 2. Load monthly files and apply project scope

In [ ]:
frames = []
loaded_files = []
missing_months = []

for month in MONTHS:
    matches = sorted(RAW_DIR.glob(f"CRMLSSold{month}*.csv"))
    if not matches:
        missing_months.append(month)
        continue

    file_path = matches[0]
    temp = pd.read_csv(file_path, low_memory=False)
    temp["source_month"] = month
    frames.append(temp)
    loaded_files.append(file_path.name)

if not frames:
    raise FileNotFoundError(f"No CRMLSSold files found in {RAW_DIR}")

df_raw = pd.concat(frames, ignore_index=True, sort=False)

print("Loaded files:", len(loaded_files))
print("Missing months:", missing_months if missing_months else "None")
print("Raw shape:", df_raw.shape)

required_scope = {"PropertyType", "PropertySubType"}
missing_scope = required_scope - set(df_raw.columns)
if missing_scope:
    raise KeyError(f"Missing scope columns: {missing_scope}")

df = df_raw[
    df_raw["PropertyType"].astype(str).str.strip().eq("Residential")
    & df_raw["PropertySubType"].astype(str).str.strip().eq("SingleFamilyResidence")
].copy()

print("SFR shape:", df.shape)

Loaded files: 17
Missing months: None
Raw shape: (363970, 81)
SFR shape: (181482, 81)


## 3. Parse dates and numeric fields

In [ ]:
date_cols = [
    "CloseDate", "ListingContractDate",
    "PurchaseContractDate", "ContractStatusChangeDate"
]
for col in date_cols:
    if col in df.columns:
        df[col] = pd.to_datetime(df[col], errors="coerce")

numeric_cols_to_parse = [
    "ClosePrice", "LivingArea", "BedroomsTotal",
    "BathroomsTotalInteger", "LotSizeSquareFeet",
    "YearBuilt", "GarageSpaces", "Stories",
    "Latitude", "Longitude"
]
for col in numeric_cols_to_parse:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors="coerce")

df["close_month"] = df["CloseDate"].dt.to_period("M").astype("string")
df["PropertyAge"] = df["CloseDate"].dt.year - df["YearBuilt"]
df["sale_month"] = df["CloseDate"].dt.month
df["month_sin"] = np.sin(2 * np.pi * df["sale_month"] / 12)
df["month_cos"] = np.cos(2 * np.pi * df["sale_month"] / 12)

df[["CloseDate", "close_month", "PropertyAge", "month_sin", "month_cos"]].head()

,CloseDate,close_month,PropertyAge,month_sin,month_cos
11,2025-01-31,2025-01,65.0,0.5,0.866025
12,2025-01-31,2025-01,53.0,0.5,0.866025
13,2025-01-31,2025-01,26.0,0.5,0.866025
14,2025-01-31,2025-01,35.0,0.5,0.866025
16,2025-01-31,2025-01,70.0,0.5,0.866025


## 4. Remove records that cannot support modeling

The notebook logs every removal. The target is never imputed.


In [ ]:
audit_rows = []

def log_step(step, before, after):
    removed = before - after
    pct = 100 * removed / before if before else 0
    audit_rows.append({
        "step": step,
        "rows_before": before,
        "rows_after": after,
        "rows_removed": removed,
        "percent_removed": pct
    })
    print(f"{step}: removed {removed:,} rows ({pct:.4f}%)")

before = len(df)
df = df[df[TARGET].notna() & df["CloseDate"].notna()].copy()
log_step("Missing target or CloseDate", before, len(df))

before = len(df)
df = df[df[TARGET] > 0].copy()
log_step("ClosePrice <= 0", before, len(df))

if "ListingContractDate" in df.columns:
    before = len(df)
    valid_date_order = (
        df["ListingContractDate"].isna()
        | (df["CloseDate"] >= df["ListingContractDate"])
    )
    df = df[valid_date_order].copy()
    log_step("CloseDate before ListingContractDate", before, len(df))

before = len(df)
if "ListingKey" in df.columns:
    df = (
        df.sort_values(["CloseDate", "source_month"])
          .drop_duplicates(subset=["ListingKey"], keep="last")
          .copy()
    )
else:
    df = df.drop_duplicates().copy()
log_step("Duplicate transactions", before, len(df))

before = len(df)
valid_living_area = df["LivingArea"].isna() | (df["LivingArea"] > 0)
df = df[valid_living_area].copy()
log_step("Non-positive LivingArea", before, len(df))

audit_log = pd.DataFrame(audit_rows)
audit_log

Missing target or CloseDate: removed 0 rows (0.0000%)
ClosePrice <= 0: removed 1 rows (0.0006%)
CloseDate before ListingContractDate: removed 29 rows (0.0160%)
Duplicate transactions: removed 133 rows (0.0733%)
Non-positive LivingArea: removed 82 rows (0.0452%)


,step,rows_before,rows_after,rows_removed,percent_removed
0,Missing target or CloseDate,181482,181482,0,0.000000
1,ClosePrice <= 0,181482,181481,1,0.000551
2,CloseDate before ListingContractDate,181481,181452,29,0.015980
3,Duplicate transactions,181452,181319,133,0.073298
4,Non-positive LivingArea,181319,181237,82,0.045224


## 5. School district spatial enrichment

In [ ]:
# Download the California School District Areas 2025-26 GeoJSON into DATA_DIR.
# The code tries to locate it automatically.

geo_candidates = []
for pattern in ["*.geojson", "*.json"]:
    geo_candidates.extend(DATA_DIR.glob(pattern))

geo_candidates = [
    p for p in geo_candidates
    if "school" in p.name.lower() or "district" in p.name.lower()
]

if not geo_candidates:
    raise FileNotFoundError(
        "School district GeoJSON not found. Download it from data.ca.gov and place it in "
        f"{DATA_DIR}. The filename should contain 'school' or 'district'."
    )

SCHOOL_GEOJSON_PATH = geo_candidates[0]
print("Using school boundary file:", SCHOOL_GEOJSON_PATH)

districts = gpd.read_file(SCHOOL_GEOJSON_PATH)

def find_column(columns, candidates):
    lower_map = {c.lower(): c for c in columns}
    for candidate in candidates:
        if candidate.lower() in lower_map:
            return lower_map[candidate.lower()]
    return None

district_type_col = find_column(
    districts.columns,
    ["DistrictType", "District_Type", "DISTRICTTYPE", "Type"]
)
district_name_col = find_column(
    districts.columns,
    ["DistrictName", "District_Name", "DISTRICTNAME", "Name"]
)

if district_type_col is None or district_name_col is None:
    print("GeoJSON columns:", districts.columns.tolist())
    raise KeyError("Could not identify DistrictType and DistrictName columns.")

districts = districts[
    districts[district_type_col].astype(str).str.strip().str.lower().eq("unified")
][[district_name_col, "geometry"]].copy()

districts = districts.rename(columns={district_name_col: "DistrictName"})
districts = districts.to_crs("EPSG:4326")

print("Unified districts:", len(districts))

Using school boundary file: /content/drive/MyDrive/IDX_Exchange_Su26/data/DistrictAreas2526_-284845464123469011(1).geojson
Unified districts: 345


In [ ]:
# Use unique coordinate pairs to make the spatial join faster.
valid_coords = df["Latitude"].notna() & df["Longitude"].notna()

coord_lookup = (
    df.loc[valid_coords, ["Latitude", "Longitude"]]
      .drop_duplicates()
      .copy()
)

property_points = gpd.GeoDataFrame(
    coord_lookup,
    geometry=gpd.points_from_xy(
        coord_lookup["Longitude"],
        coord_lookup["Latitude"]
    ),
    crs="EPSG:4326"
)

coord_district = gpd.sjoin(
    property_points,
    districts,
    how="left",
    predicate="within"
).drop(columns=["geometry", "index_right"], errors="ignore")

df = df.merge(
    coord_district,
    on=["Latitude", "Longitude"],
    how="left"
)

df["DistrictName"] = df["DistrictName"].fillna("Unknown")

print(df["DistrictName"].value_counts(dropna=False).head())
print("District mapping coverage:",
      f"{100 * (df['DistrictName'] != 'Unknown').mean():.2f}%")

DistrictName
Unknown                 43955
Los Angeles Unified     18068
San Diego Unified        4943
Desert Sands Unified     3489
Capistrano Unified       3340
Name: count, dtype: int64
District mapping coverage: 75.75%


## 6. Leakage audit and model feature configuration

`ListPrice`, `OriginalListPrice`, `DaysOnMarket`, post-close fields, identifiers, agent fields, and variables derived from the target are excluded from model inputs.


In [ ]:
LEAKAGE_OR_UNSTABLE_COLUMNS = [
    TARGET,
    "ListPrice",
    "OriginalListPrice",
    "ClosePrice_to_ListPrice_ratio",
    "DaysOnMarket",
    "CloseDate",
    "PurchaseContractDate",
    "ContractStatusChangeDate",
    "MlsStatus",
    "ListingKey",
    "ListingKeyNumeric",
    "ListingId",
    "ListAgentEmail",
    "ListAgentFirstName",
    "ListAgentLastName",
    "ListAgentFullName",
    "BuyerAgentFirstName",
    "BuyerAgentLastName",
    "BuyerAgentMlsId",
    "ListOfficeName",
    "BuyerOfficeName",
    "CoListOfficeName",
    "UnparsedAddress"
]

numeric_features = [
    "LivingArea",
    "BedroomsTotal",
    "BathroomsTotalInteger",
    "LotSizeSquareFeet",
    "PropertyAge",
    "GarageSpaces",
    "Stories",
    "Latitude",
    "Longitude",
    "month_sin",
    "month_cos"
]

categorical_features = [
    "City",
    "PostalCode",
    "CountyOrParish",
    "DistrictName",
    "ViewYN",
    "WaterfrontYN",
    "BasementYN",
    "PoolPrivateYN",
    "AttachedGarageYN",
    "FireplaceYN",
    "NewConstructionYN"
]

numeric_features = [c for c in numeric_features if c in df.columns]
categorical_features = [c for c in categorical_features if c in df.columns]
feature_columns = numeric_features + categorical_features

leakage_selected = sorted(set(feature_columns) & set(LEAKAGE_OR_UNSTABLE_COLUMNS))
if leakage_selected:
    raise ValueError(f"Leakage columns selected: {leakage_selected}")

print("Numeric features:", numeric_features)
print("Categorical features:", categorical_features)
print("Total raw features:", len(feature_columns))

Numeric features: ['LivingArea', 'BedroomsTotal', 'BathroomsTotalInteger', 'LotSizeSquareFeet', 'PropertyAge', 'GarageSpaces', 'Stories', 'Latitude', 'Longitude', 'month_sin', 'month_cos']
Categorical features: ['City', 'PostalCode', 'CountyOrParish', 'DistrictName', 'ViewYN', 'WaterfrontYN', 'BasementYN', 'PoolPrivateYN', 'AttachedGarageYN', 'FireplaceYN', 'NewConstructionYN']
Total raw features: 22


## 7. Missing-value summary and missing indicators

In [ ]:
missing_summary = pd.DataFrame({
    "missing_count": df[feature_columns + [TARGET]].isna().sum(),
    "missing_percent": 100 * df[feature_columns + [TARGET]].isna().mean(),
    "dtype": df[feature_columns + [TARGET]].dtypes.astype(str)
}).sort_values("missing_percent", ascending=False)

missing_summary.to_csv(OUTPUT_DIR / "missing_summary.csv")
missing_summary

,missing_count,missing_percent,dtype
WaterfrontYN,181141,99.947031,object
BasementYN,176865,97.587689,object
AttachedGarageYN,21729,11.989274,object
Stories,19542,10.782566,float64
ViewYN,16238,8.959539,object
PoolPrivateYN,14361,7.923879,object
NewConstructionYN,13700,7.559163,object
GarageSpaces,7055,3.892693,float64
LotSizeSquareFeet,3160,1.743573,float64
FireplaceYN,133,0.073385,object


In [ ]:
# Missingness may contain information. Add indicators for selected important numeric features.
missing_indicator_source_cols = [
    c for c in [
        "LivingArea",
        "BathroomsTotalInteger",
        "LotSizeSquareFeet",
        "YearBuilt",
        "GarageSpaces",
        "Stories"
    ]
    if c in df.columns and df[c].isna().any()
]

for col in missing_indicator_source_cols:
    flag_col = f"{col}_missing"
    df[flag_col] = df[col].isna().astype("int8")
    numeric_features.append(flag_col)

feature_columns = numeric_features + categorical_features

print("Missing indicators added:", missing_indicator_source_cols)

Missing indicators added: ['LivingArea', 'BathroomsTotalInteger', 'LotSizeSquareFeet', 'YearBuilt', 'GarageSpaces', 'Stories']


## 8. Chronological train/validation/test split

- Test: most recent complete month.
- Validation: second-most-recent complete month.
- Training: the X months immediately preceding validation.
- X is selected later using validation performance, never test performance.


In [ ]:
available_months = sorted(df["close_month"].dropna().unique())
test_month = available_months[-1]
validation_month = available_months[-2]

test_period = pd.Period(test_month, freq="M")
validation_period = pd.Period(validation_month, freq="M")
train_end_period = validation_period - 1

split_rows = []
month_period = pd.PeriodIndex(df["close_month"], freq="M")

for window in TRAINING_WINDOW_CANDIDATES:
    train_start_period = train_end_period - (window - 1)
    train_mask = (
        (month_period >= train_start_period)
        & (month_period <= train_end_period)
    )
    validation_mask = month_period == validation_period
    test_mask = month_period == test_period

    split_rows.append({
        "training_window_months": window,
        "train_start_month": str(train_start_period),
        "train_end_month": str(train_end_period),
        "validation_month": str(validation_period),
        "test_month": str(test_period),
        "train_rows": int(train_mask.sum()),
        "validation_rows": int(validation_mask.sum()),
        "test_rows": int(test_mask.sum())
    })

split_window_summary = pd.DataFrame(split_rows)
split_window_summary.to_csv(
    OUTPUT_DIR / "training_window_candidates.csv",
    index=False
)
split_window_summary

,training_window_months,train_start_month,train_end_month,validation_month,test_month,train_rows,validation_rows,test_rows
0,3,2026-01,2026-03,2026-04,2026-05,27185,12019,12020
1,6,2025-10,2026-03,2026-04,2026-05,59360,12019,12020
2,9,2025-07,2026-03,2026-04,2026-05,94343,12019,12020
3,12,2025-04,2026-03,2026-04,2026-05,129636,12019,12020
4,15,2025-01,2026-03,2026-04,2026-05,157198,12019,12020


In [ ]:
# Default split for producing preprocessing artifacts.
# The optimal X must be selected later from validation results.
train_start_period = train_end_period - (DEFAULT_TRAINING_WINDOW - 1)

train_mask = (
    (month_period >= train_start_period)
    & (month_period <= train_end_period)
)
validation_mask = month_period == validation_period
test_mask = month_period == test_period

train_df = df.loc[train_mask].copy()
validation_df = df.loc[validation_mask].copy()
test_df = df.loc[test_mask].copy()

print("Train:", train_df["close_month"].min(), "to", train_df["close_month"].max(), len(train_df))
print("Validation:", validation_month, len(validation_df))
print("Test:", test_month, len(test_df))

Train: 2025-04 to 2026-03 129636
Validation: 2026-04 12019
Test: 2026-05 12020


## 9. Learn ClosePrice outlier thresholds from training data only

The 0.5th and 99.5th percentile thresholds are computed on training data and then frozen.


In [ ]:
lower_price = train_df[TARGET].quantile(0.005)
upper_price = train_df[TARGET].quantile(0.995)

print(f"Training-derived lower threshold: ${lower_price:,.2f}")
print(f"Training-derived upper threshold: ${upper_price:,.2f}")

def apply_frozen_target_bounds(frame):
    return frame[frame[TARGET].between(lower_price, upper_price)].copy()

train_df = apply_frozen_target_bounds(train_df)
validation_df = apply_frozen_target_bounds(validation_df)
test_df = apply_frozen_target_bounds(test_df)

print("After frozen target bounds:")
print("Train:", len(train_df))
print("Validation:", len(validation_df))
print("Test:", len(test_df))

Training-derived lower threshold: $185,000.00
Training-derived upper threshold: $8,800,000.00
After frozen target bounds:
Train: 128362
Validation: 11897
Test: 11910


## 10. Leakage-safe encoding, imputation, and scaling

The preprocessing object is fit on training data only. Validation and test data are transformed using the unchanged fitted object.


In [ ]:
numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(
        strategy="median",
        add_indicator=False
    )),
    ("scaler", StandardScaler())
])

categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(
        handle_unknown="ignore",
        drop="first",
        min_frequency=20
    ))
])

preprocessor = ColumnTransformer([
    ("numeric", numeric_pipeline, numeric_features),
    ("categorical", categorical_pipeline, categorical_features)
])

X_train = train_df[feature_columns].copy()
y_train = train_df[TARGET].copy()

X_validation = validation_df[feature_columns].copy()
y_validation = validation_df[TARGET].copy()

X_test = test_df[feature_columns].copy()
y_test = test_df[TARGET].copy()

X_train_processed = preprocessor.fit_transform(X_train)
X_validation_processed = preprocessor.transform(X_validation)
X_test_processed = preprocessor.transform(X_test)

print("Train processed shape:", X_train_processed.shape)
print("Validation processed shape:", X_validation_processed.shape)
print("Test processed shape:", X_test_processed.shape)

Train processed shape: (128362, 1626)
Validation processed shape: (11897, 1626)
Test processed shape: (11910, 1626)


## 11. Save cleaned data and reusable preprocessing artifacts

In [ ]:
# Save the human-readable cleaned/enriched data.
# Do not save a globally imputed or globally scaled CSV, because those parameters
# must depend on the chosen training period.
CLEANED_DATA_PATH = DATA_DIR / "crmls_sfr_avm_cleaned_enriched_202501_202605.csv"
df.to_csv(CLEANED_DATA_PATH, index=False)

# Save split-specific human-readable CSVs.
train_df.to_csv(OUTPUT_DIR / "train_default_12_month.csv", index=False)
validation_df.to_csv(OUTPUT_DIR / "validation_202604.csv", index=False)
test_df.to_csv(OUTPUT_DIR / "test_202605.csv", index=False)

# Save the preprocessing object and metadata.
joblib.dump(preprocessor, OUTPUT_DIR / "preprocessor_default_12_month.joblib")

metadata = {
    "target": TARGET,
    "date_column": DATE_COL,
    "default_training_window_months": DEFAULT_TRAINING_WINDOW,
    "train_start_month": str(train_start_period),
    "train_end_month": str(train_end_period),
    "validation_month": str(validation_period),
    "test_month": str(test_period),
    "numeric_features": numeric_features,
    "categorical_features": categorical_features,
    "target_lower_bound_from_train": float(lower_price),
    "target_upper_bound_from_train": float(upper_price),
    "random_seed": RANDOM_SEED
}

with open(OUTPUT_DIR / "preprocessing_metadata.json", "w") as f:
    json.dump(metadata, f, indent=2)

audit_log.to_csv(OUTPUT_DIR / "cleaning_audit_log.csv", index=False)

print("Saved cleaned dataset:", CLEANED_DATA_PATH)
print("Saved outputs to:", OUTPUT_DIR)

Saved cleaned dataset: /content/drive/MyDrive/IDX_Exchange_Su26/data/crmls_sfr_avm_cleaned_enriched_202501_202605.csv
Saved outputs to: /content/drive/MyDrive/IDX_Exchange_Su26/data/week3_outputs


## 12. Final checks

The cleaned CSV is intentionally not permanently one-hot encoded or standardized. Those transformations depend on the training window and are therefore stored in the fitted preprocessing pipeline. This prevents validation/test leakage and lets Weeks 4 and 5 reuse the exact same preprocessing.


In [ ]:
assert not any(c in feature_columns for c in [
    "ListPrice",
    "OriginalListPrice",
    "DaysOnMarket",
    "ClosePrice_to_ListPrice_ratio"
])

assert train_df["close_month"].max() < validation_month
assert validation_month < test_month

print("Leakage audit passed.")
print("Chronological split passed.")
print("Target missing values:", df[TARGET].isna().sum())
print("Final cleaned shape:", df.shape)
print("Feature count before encoding:", len(feature_columns))

Leakage audit passed.
Chronological split passed.
Target missing values: 0
Final cleaned shape: (181237, 93)
Feature count before encoding: 28
